# Cross-Benchmark Performance Evaluation

This notebook establishes the comparative performance metrics for the project using standard statistical controls:
1. **BCE Loss:** Binary Cross-Entropy is used as the objective function for all classification models.
2. **Probability Mapping:** Quantum expectation values (Pauli-Z) are mapped from `[-1, 1]` to the probability domain `[0, 1]`.
3. **Statistical Averaging:** Results are reported as the mean and standard deviation across 5 random initialization seeds.

In [ ]:
import sys
sys.path.append('..')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from utils.data_utils import load_higgs, binary_accuracy

SEEDS = [42, 7, 123, 1, 99]
N_SAMPLES = 1000
N_FEATURES = 8
PATH = '../data/HIGGS.csv.gz'

## 1. Classical MLP Benchmark
Training the Strong MLP across 5 seeds using Log-Loss.

In [ ]:
classical_aucs = []

for seed in SEEDS:
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, 1)
    )
    
    model = MLPClassifier(
        hidden_layer_sizes=(100, 50, 25), 
        max_iter=500, 
        early_stopping=True, 
        random_state=seed
    )
    model.fit(X_tr, y_tr)
    
    probs = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, probs)
    classical_aucs.append(auc)
    print(f"Seed {seed} | Classical Test AUC: {auc:.4f}")

c_mean = np.mean(classical_aucs)
c_std = np.std(classical_aucs)

## 2. Quantum VQC Benchmark
Training the Synthesis 2.1 configuration using Binary Cross-Entropy loss.

In [ ]:
dev = qml.device('lightning.qubit', wires=N_FEATURES)

@qml.qnode(dev, interface='autograd')
def circuit(weights, x):
    for l in range(3):
        for i in range(len(x)): qml.RY(x[i], wires=i)
        for q in range(len(x)): qml.Rot(*weights[l, q], wires=q)
        for q in range(len(x)): qml.CNOT(wires=[q, (q + 1) % len(x)])
    return qml.expval(qml.PauliZ(0))

def vqc_prob(w, x, b):
    return (circuit(w, x) + b + 1.0) / 2.0

def bce_loss(weights, bias, X, y):
    probs = pnp.array([vqc_prob(weights, x, bias) for x in X])
    probs = pnp.clip(probs, 1e-15, 1.0 - 1e-15)
    return -pnp.mean(y * pnp.log(probs) + (1 - y) * pnp.log(1 - probs))

def train_quantum(seed):
    X_tr, X_vl, X_te, y_tr, y_vl, y_te = load_higgs(
        path=PATH, n_samples=N_SAMPLES, n_features=N_FEATURES, random_state=seed, scale_range=(0, np.pi)
    )
    pnp.random.seed(seed)
    w = pnp.array(pnp.random.uniform(0, 2*np.pi, (3, N_FEATURES, 3)), requires_grad=True)
    b = pnp.array(0.0, requires_grad=True)
    opt = qml.AdamOptimizer(stepsize=0.05)
    
    for epoch in range(40):
        idx = np.random.permutation(len(X_tr))
        for start in range(0, len(X_tr), 32):
            Xb, yb = X_tr[idx][start:start+32], y_tr[idx][start:start+32].astype(float)
            w, b = opt.step(lambda w_, b_: bce_loss(w_, b_, Xb, yb), w, b)
            
    test_probs = np.array([float(vqc_prob(w, x, b)) for x in X_te])
    return roc_auc_score(y_te, test_probs)

quantum_aucs = []
for seed in SEEDS:
    auc = train_quantum(seed)
    quantum_aucs.append(auc)
    print(f"Seed {seed} | Quantum Test AUC: {auc:.4f}")

q_mean = np.mean(quantum_aucs)
q_std = np.std(quantum_aucs)

## 3. Comparative Summary

In [ ]:
print(f"\nClassical Result: {c_mean:.4f} ± {c_std:.4f}")
print(f"Quantum Result:   {q_mean:.4f} ± {q_std:.4f}")

plt.bar(['Classical', 'Quantum'], [c_mean, q_mean], yerr=[c_std, q_std], capsize=10)
plt.ylabel('Mean Test AUC')
plt.title('Final Benchmarking Comparison')
plt.ylim(0.5, 0.7)
plt.show()